# Capital allocation: stipulated synthetic compatibility computation

Uses-of-cash shares and the value target are stipulated. Nothing below estimates causal value creation, future returns, a regime, or a motion operator.

The legacy compatibility pipeline is scientifically flawed as a detector, has no validated detection capability, and is not a mixture-motion estimator. A centered-log-ratio motion-estimator repair is designed but is not implemented in this notebook. The computations below expose compatibility behavior only.

NMF is a full-sample reconstruction tool here. The same nonnegative observation matrix is passed to NMF and membership re-inference. Every R2 is in-sample. Equilibrium records describe only the fitted matrix; they are not findings about the data-generating process.


In [ ]:
# Imports and deterministic seed
import os
import random
import numpy as np
import ts2eg as gm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
TS2EG_CI = os.getenv("TS2EG_CI", "0") == "1"


## Stipulated allocation shares


In [ ]:
rng = np.random.default_rng(SEED)
T = 64
t = np.arange(T)
base = np.array([0.54, 0.20, 0.08, 0.10, 0.08])[:, None]
variation = np.vstack([
    0.025 * np.sin(2 * np.pi * t / 20),
    0.018 * np.cos(2 * np.pi * (t + 2) / 16),
    0.012 * np.sin(2 * np.pi * (t + 5) / 24),
    0.015 * np.cos(2 * np.pi * (t + 1) / 28),
    0.012 * np.sin(2 * np.pi * (t + 3) / 18),
])
X_model = np.maximum(base + variation + 0.004 * rng.standard_normal((5, T)), 1e-8)
X_model /= X_model.sum(axis=0, keepdims=True)
value_target = (
    -0.5 * X_model[0] - 0.2 * X_model[1]
    + 0.6 * X_model[2] + 0.3 * X_model[3]
    + 0.03 * rng.standard_normal(T)
)
print("allocation-share matrix:", X_model.shape)


## Time-constant value-gradient compatibility behavior

value_gradient_payoffs fits one full-sample ridge coefficient vector and repeats it at every time. Constancy and a high in-sample fit are mechanical compatibility behavior, not evidence of economic stability.


In [ ]:
v_value = gm.value_gradient_payoffs(
    X_model, value_target, model="ridge", ridge=1e-2, standardize=True
)
S, H = gm.nmf_on_X(X_model, k=3, iters=120, seed=SEED, normalize="l2")
fit = gm.estimate_A_from_series(S, X_model, v_value, k=3, ridge=1e-2)
records = gm.find_ESS(fit["A"], max_support=3)
summary = [
    (r["support"], r["ess_status"], r["stability_status"])
    for r in records if r["is_nash"]
]
print("Maximum within-row payoff range:", float(np.max(np.ptp(v_value, axis=1))))
print("Legacy descriptive payoff-field R2 (in-sample):", round(fit["R2"], 3))
print("Fitted-matrix Nash/ESS/stability descriptions:", summary)
